## 03 – Isochronen mit OpenRouteService (ORS)

Ziel: Fahrzeit-/Gehzeit-Isochronen für Apotheken in der Schweiz berechnen und auswerten.

**Kontext**
- Eingabe: `data/processed/pharmacies_ch.geojson` (EPSG:2056)
- Wir nutzen die **public ORS API** (Key erforderlich). Key gibt’s kostenlos bei: `https://openrouteservice.org/dev/#/signup`

Wichtig: ORS erwartet Koordinaten in **WGS84 (EPSG:4326)**. Wir transformieren daher von LV95 → WGS84 und nach der Abfrage wieder zurück nach LV95.

In [ ]:
import time

# Nur 20 Apotheken als Stichprobe (Rate Limit: 40 req/min)
SAMPLE_SIZE = 20
gdf_sample = gdf_pharm.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Stichprobe: {SAMPLE_SIZE} von {len(gdf_pharm)} Apotheken")

results = []
ranges_s = [300, 600, 900]  # 5, 10, 15 Minuten

for i, row in gdf_sample.iterrows():
    pt_lv95 = row.geometry
    pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
    lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)
    try:
        res = ors_isochrones("driving-car", lon, lat, ranges_s)
        for feat in res.get("features", []):
            feat["properties"]["pharmacy_id"] = row.get("osm_id", i)
            results.append(feat)
        print(f"{i+1}/{SAMPLE_SIZE} OK")
    except Exception as e:
        if "429" in str(e):
            print(f"{i+1}/{SAMPLE_SIZE} Rate limit - warte 15s...")
            time.sleep(15)
        else:
            print(f"{i+1}/{SAMPLE_SIZE} Fehler: {e}")
    time.sleep(2)  # 2s zwischen Requests

print(f"Isochronen berechnet: {len(results)} Features")


In [8]:
# 2) Isochrone für eine einzelne Apotheke testen (5/10/15min zu Fuss + Auto)

pharm_path = PROCESSED / "pharmacies_ch.geojson"
gdf_pharm = gpd.read_file(pharm_path)
if gdf_pharm.crs is None:
    gdf_pharm = gdf_pharm.set_crs(2056)

print("Apotheken geladen:", len(gdf_pharm), "CRS:", gdf_pharm.crs)

# Nimm eine Apotheke (erste Zeile). ORS braucht WGS84 lon/lat.
row = gdf_pharm.iloc[0]
geom = row.geometry
pt_lv95 = geom if geom.geom_type == "Point" else geom.centroid
pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)

ranges_s = [5 * 60, 10 * 60, 15 * 60]

print("Testpunkt WGS84:", (lon, lat))
print("Abfrage: foot-walking...")
res_walk = ors_isochrones("foot-walking", lon, lat, ranges_s)
print("OK – Features (walk):", len(res_walk.get("features", [])))

print("Abfrage: driving-car...")
res_car = ors_isochrones("driving-car", lon, lat, ranges_s)
print("OK – Features (car):", len(res_car.get("features", [])))

# Konvertiere zu GeoDataFrame für schnellen Plausibilitätscheck
gdf_walk = gpd.GeoDataFrame.from_features(res_walk["features"], crs=4326).to_crs(2056)
gdf_car = gpd.GeoDataFrame.from_features(res_car["features"], crs=4326).to_crs(2056)

print("Beispiel (walk) Spalten:", list(gdf_walk.columns))
print("Beispiel (car)  Spalten:", list(gdf_car.columns))
print("Fläche (car) in km² pro Range (nur Test):")
print((gdf_car.area / 1_000_000).round(2).to_string(index=False))


Apotheken geladen: 1640 CRS: EPSG:2056
Testpunkt WGS84: (8.726227708612216, 47.24097511006537)
Abfrage: foot-walking...


RuntimeError: ORS_API_KEY fehlt. Bitte setze ihn als Umgebungsvariable ORS_API_KEY oder in ../.env (ORS_API_KEY=...).

In [ ]:
import time

# Nur 20 Apotheken als Stichprobe (Rate Limit: 40 req/min)
SAMPLE_SIZE = 20
gdf_sample = gdf_pharm.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Stichprobe: {SAMPLE_SIZE} von {len(gdf_pharm)} Apotheken")

results = []
ranges_s = [300, 600, 900]  # 5, 10, 15 Minuten

for i, row in gdf_sample.iterrows():
    pt_lv95 = row.geometry
    pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
    lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)
    try:
        res = ors_isochrones("driving-car", lon, lat, ranges_s)
        for feat in res.get("features", []):
            feat["properties"]["pharmacy_id"] = row.get("osm_id", i)
            results.append(feat)
        print(f"{i+1}/{SAMPLE_SIZE} OK")
    except Exception as e:
        if "429" in str(e):
            print(f"{i+1}/{SAMPLE_SIZE} Rate limit - warte 15s...")
            time.sleep(15)
        else:
            print(f"{i+1}/{SAMPLE_SIZE} Fehler: {e}")
    time.sleep(2)  # 2s zwischen Requests

print(f"Isochronen berechnet: {len(results)} Features")


In [ ]:
import time

# Nur 20 Apotheken als Stichprobe (Rate Limit: 40 req/min)
SAMPLE_SIZE = 20
gdf_sample = gdf_pharm.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Stichprobe: {SAMPLE_SIZE} von {len(gdf_pharm)} Apotheken")

results = []
ranges_s = [300, 600, 900]  # 5, 10, 15 Minuten

for i, row in gdf_sample.iterrows():
    pt_lv95 = row.geometry
    pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
    lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)
    try:
        res = ors_isochrones("driving-car", lon, lat, ranges_s)
        for feat in res.get("features", []):
            feat["properties"]["pharmacy_id"] = row.get("osm_id", i)
            results.append(feat)
        print(f"{i+1}/{SAMPLE_SIZE} OK")
    except Exception as e:
        if "429" in str(e):
            print(f"{i+1}/{SAMPLE_SIZE} Rate limit - warte 15s...")
            time.sleep(15)
        else:
            print(f"{i+1}/{SAMPLE_SIZE} Fehler: {e}")
    time.sleep(2)  # 2s zwischen Requests

print(f"Isochronen berechnet: {len(results)} Features")


In [ ]:
# 5) Karte: Isochronen über Gemeindegrenzen, Apotheken als Punkte

gdf_gem_plot = gdf_gem.to_crs(2056)
gdf_pharm_plot = gdf_pharm.to_crs(2056)

# Falls Isochronen (5min car) nicht im Speicher sind, lade aus Datei
if "gdf_iso_5_car" not in globals():
    out_gpkg = (PROCESSED / "isochrones_5min_car.gpkg").resolve()
    gdf_iso_5_car = gpd.read_file(out_gpkg, layer="isochrones_5min_car")
    if gdf_iso_5_car.crs is None:
        gdf_iso_5_car = gdf_iso_5_car.set_crs(2056)

out_png = (OUTPUTS / "03_isochronen_5min_auto.png").resolve()

fig, ax = plt.subplots(figsize=(14, 10))
gdf_gem_plot.boundary.plot(ax=ax, linewidth=0.25, color="#666666", alpha=0.6)

# Isochronen (5min Auto)
gdf_iso_5_car.plot(ax=ax, color="#ff6b6b", alpha=0.18, edgecolor="none")

# Apothekenpunkte
gdf_pharm_plot.plot(ax=ax, color="#1f3a93", markersize=3, alpha=0.7)

ax.set_title("Isochronen 5 Minuten (Auto) zu Apotheken", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()

print("Karte gespeichert:", out_png)
print("Layer:")
print("- Gemeinden:", len(gdf_gem_plot))
print("- Apotheken:", len(gdf_pharm_plot))
print("- Isochronen 5min Auto:", len(gdf_iso_5_car))


In [ ]:
# 6) Ergebnisse speichern in data/processed/

# Speichere Isochronen (5min Auto) zusätzlich als GeoJSON (handlich, aber größer)
out_geojson = (PROCESSED / "isochrones_5min_car.geojson").resolve()
try:
    gdf_iso_5_car.to_file(out_geojson, driver="GeoJSON")
    print("Isochronen GeoJSON gespeichert:", out_geojson)
except Exception as e:
    print("Konnte GeoJSON nicht schreiben (evtl. Dateisperre/Encoding):", e)

# Speichere Union-Summary (falls vorhanden)
out_union_csv = (PROCESSED / "isochronen_union_summary.csv").resolve()
if "df_union" in globals():
    df_union.to_csv(out_union_csv, index=False)
    print("Union-Summary CSV gespeichert:", out_union_csv)
else:
    print("Union-Summary nicht vorhanden (Zelle 4 ggf. noch nicht ausgeführt).")

print("Fertig – Outputs liegen in data/processed/ und outputs/maps/")
